In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Galera do Bem - P2/dados', exist_ok=True)
os.chdir('/content/drive/MyDrive/Galera do Bem - P2')
print("Pasta de trabalho:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta de trabalho: /content/drive/MyDrive/Galera do Bem - P2


In [ ]:
%%writefile tabelas.sql
-- ============================================================
-- Projeto Integrador II - UEPB
-- Equipe: Galera do Bem
-- Base: DATASUS - PO - Painel de Oncologia - 2022
-- Modelo: Caso agregado
-- ============================================================

CREATE TABLE tempo (
    id_tempo INTEGER PRIMARY KEY,
    data     TEXT NOT NULL UNIQUE,
    ano      INTEGER NOT NULL,
    mes      INTEGER NOT NULL CHECK (mes BETWEEN 1 AND 12),
    dia      INTEGER NOT NULL CHECK (dia BETWEEN 1 AND 31)
);

CREATE TABLE municipio (
    cod_mun  TEXT PRIMARY KEY,
    cod_uf   TEXT NOT NULL,
    nome_uf  TEXT
);

CREATE TABLE caso_cancer (
    id_caso            INTEGER PRIMARY KEY,
    id_tempo_diag      INTEGER NOT NULL REFERENCES tempo(id_tempo),
    id_tempo_trat      INTEGER          REFERENCES tempo(id_tempo),
    mun_diag           TEXT NOT NULL REFERENCES municipio(cod_mun),
    cnes_diag          TEXT NOT NULL,
    diag_deth          TEXT,
    idade              INTEGER,
    sexo               TEXT CHECK (sexo IN ('M','F','I')),
    estadiamento       INTEGER,
    qtd_procedimentos  INTEGER,
    tempo_trat         INTEGER,
    dt_nasc            TEXT,
    UNIQUE (id_tempo_diag, mun_diag, cnes_diag, diag_deth, idade, sexo)
);

Overwriting tabelas.sql


In [ ]:
%%writefile carga.sql
-- ============================================================
-- Passo 3 da carga: povoar as tabelas modeladas
-- ============================================================

-- 2.1 Popular tempo
INSERT OR IGNORE INTO tempo (data, ano, mes, dia)
SELECT DISTINCT
    substr(dt, 7, 4) || '-' || substr(dt, 4, 2) || '-' || substr(dt, 1, 2) AS data_iso,
    CAST(substr(dt, 7, 4) AS INTEGER),
    CAST(substr(dt, 4, 2) AS INTEGER),
    CAST(substr(dt, 1, 2) AS INTEGER)
FROM (
    SELECT dt_diag AS dt FROM bruto_cancer WHERE dt_diag IS NOT NULL AND TRIM(dt_diag) <> ''
    UNION
    SELECT dt_trat AS dt FROM bruto_cancer WHERE dt_trat IS NOT NULL AND TRIM(dt_trat) <> ''
    UNION
    SELECT dt_nasc AS dt FROM bruto_cancer WHERE dt_nasc IS NOT NULL AND TRIM(dt_nasc) <> ''
)
WHERE LENGTH(dt) = 10 AND substr(dt, 3, 1) = '/' AND substr(dt, 6, 1) = '/';

-- 2.2 Popular municipio
INSERT OR IGNORE INTO municipio (cod_mun, cod_uf, nome_uf)
SELECT DISTINCT
    printf('%06d', CAST(mun_diag AS INTEGER)),
    printf('%02d', CAST(uf_diagn AS INTEGER)),
    NULL
FROM bruto_cancer
WHERE mun_diag IS NOT NULL AND TRIM(mun_diag) <> ''
  AND uf_diagn IS NOT NULL AND TRIM(uf_diagn) <> '';

-- 2.3 Popular caso_cancer (agregando por caso)
INSERT INTO caso_cancer (
    id_tempo_diag, id_tempo_trat, mun_diag, cnes_diag,
    diag_deth, idade, sexo, estadiamento,
    qtd_procedimentos, tempo_trat, dt_nasc
)
SELECT
    td.id_tempo,
    tt.id_tempo,
    printf('%06d', CAST(b.mun_diag AS INTEGER)),
    printf('%d',   CAST(b.cnes_diag AS INTEGER)),
    b.diag_deth,
    CAST(b.idade AS INTEGER),
    b.sexo,
    CASE
        WHEN b.estadiam IS NULL OR TRIM(b.estadiam) = '' THEN NULL
        WHEN CAST(b.estadiam AS INTEGER) = 9 THEN NULL
        ELSE CAST(b.estadiam AS INTEGER)
    END,
    COUNT(*),
    MIN(
        CASE
            WHEN b.tempo_trat IS NULL OR TRIM(b.tempo_trat) = '' THEN NULL
            WHEN CAST(b.tempo_trat AS INTEGER) = 99999 THEN NULL
            WHEN CAST(b.tempo_trat AS INTEGER) < 0 THEN NULL
            ELSE CAST(b.tempo_trat AS INTEGER)
        END
    ),
    CASE
        WHEN b.dt_nasc IS NULL OR TRIM(b.dt_nasc) = '' THEN NULL
        ELSE substr(b.dt_nasc, 7, 4) || '-' || substr(b.dt_nasc, 4, 2) || '-' || substr(b.dt_nasc, 1, 2)
    END
FROM (
    SELECT *,
        CASE
            WHEN dt_trat IS NULL OR TRIM(dt_trat) = '' THEN NULL
            ELSE substr(dt_trat, 7, 4) || '-' || substr(dt_trat, 4, 2) || '-' || substr(dt_trat, 1, 2)
        END AS dt_trat_iso
    FROM bruto_cancer
) b
LEFT JOIN tempo td
       ON td.data = substr(b.dt_diag, 7, 4) || '-' || substr(b.dt_diag, 4, 2) || '-' || substr(b.dt_diag, 1, 2)
LEFT JOIN tempo tt
       ON tt.data = b.dt_trat_iso
WHERE b.dt_diag IS NOT NULL AND TRIM(b.dt_diag) <> ''
  AND b.mun_diag IS NOT NULL AND TRIM(b.mun_diag) <> ''
  AND b.cnes_diag IS NOT NULL AND TRIM(b.cnes_diag) <> ''
GROUP BY
    td.id_tempo, tt.id_tempo, b.mun_diag, b.cnes_diag,
    b.diag_deth, b.idade, b.sexo, b.estadiam;

Overwriting carga.sql


In [ ]:
%%writefile verificacoes.sql
-- 1) Total no staging vs total no modelo
SELECT
    (SELECT COUNT(*) FROM bruto_cancer) AS staging,
    (SELECT COUNT(*) FROM caso_cancer)  AS casos_agregados;

-- 2) CNS
SELECT COUNT(*) AS cns_nulos
FROM bruto_cancer
WHERE cns_pac IS NULL OR TRIM(cns_pac) = '';

-- 3) Órfãos
SELECT COUNT(*) AS orfaos_municipio
FROM caso_cancer c
LEFT JOIN municipio m ON c.mun_diag = m.cod_mun
WHERE m.cod_mun IS NULL;

-- 4) Nulos por coluna
SELECT
    SUM(CASE WHEN id_tempo_trat IS NULL THEN 1 ELSE 0 END) AS sem_data_trat,
    SUM(CASE WHEN estadiamento  IS NULL THEN 1 ELSE 0 END) AS sem_estadiamento,
    SUM(CASE WHEN tempo_trat     IS NULL THEN 1 ELSE 0 END) AS sem_tempo_trat
FROM caso_cancer;

-- 5) Grão
SELECT id_tempo_diag, mun_diag, cnes_diag, diag_deth, idade, sexo, COUNT(*) AS qtd
FROM caso_cancer
GROUP BY id_tempo_diag, mun_diag, cnes_diag, diag_deth, idade, sexo
HAVING COUNT(*) > 1
LIMIT 20;

-- 6) Valores impossíveis
SELECT
    MIN(idade) AS idade_min, MAX(idade) AS idade_max,
    MIN(tempo_trat) AS tempo_min, MAX(tempo_trat) AS tempo_max,
    MIN(qtd_procedimentos) AS qtd_min, MAX(qtd_procedimentos) AS qtd_max
FROM caso_cancer;

Overwriting verificacoes.sql


In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv(
    "/content/drive/MyDrive/POBR2022.csv",   # ajuste o caminho
    sep=',', encoding='utf-8-sig',
    dtype=str, na_values=['', '-', 'NA', '.']
)
df.columns = df.columns.str.strip().str.replace(' ', '_').str.upper()

os.makedirs('dados', exist_ok=True)
con = sqlite3.connect('dados/base.db')

df.to_sql('bruto_cancer', con, if_exists='replace', index=False)
con.commit()

print("Staging carregado:")
print(pd.read_sql_query("SELECT COUNT(*) AS linhas FROM bruto_cancer", con))

Staging carregado:
   linhas
0  635713


In [ ]:
# ============================================================
# Célula 6 — Criar tabelas, povoar o modelo (Opção 2)
# ============================================================

import sqlite3
import pandas as pd

con = sqlite3.connect('dados/base.db')

# ------------------------------------------------------------
# 1. Derrubar tabelas modeladas, se existirem (staging intacto)
# ------------------------------------------------------------
con.executescript("""
    DROP TABLE IF EXISTS caso_cancer;
    DROP TABLE IF EXISTS municipio;
    DROP TABLE IF EXISTS tempo;
""")

# ------------------------------------------------------------
# 2. Criar tabelas
# ------------------------------------------------------------
con.executescript(open('tabelas.sql').read())
print("Tabelas criadas.")

# ------------------------------------------------------------
# 3. Povoar tempo
# ------------------------------------------------------------
con.executescript("""
INSERT OR IGNORE INTO tempo (data, ano, mes, dia)
SELECT DISTINCT
    substr(dt, 7, 4) || '-' || substr(dt, 4, 2) || '-' || substr(dt, 1, 2),
    CAST(substr(dt, 7, 4) AS INTEGER),
    CAST(substr(dt, 4, 2) AS INTEGER),
    CAST(substr(dt, 1, 2) AS INTEGER)
FROM (
    SELECT dt_diag AS dt FROM bruto_cancer WHERE dt_diag IS NOT NULL AND TRIM(dt_diag) <> ''
    UNION
    SELECT dt_trat AS dt FROM bruto_cancer WHERE dt_trat IS NOT NULL AND TRIM(dt_trat) <> ''
    UNION
    SELECT dt_nasc AS dt FROM bruto_cancer WHERE dt_nasc IS NOT NULL AND TRIM(dt_nasc) <> ''
)
WHERE LENGTH(dt) = 10
  AND substr(dt, 3, 1) = '/'
  AND substr(dt, 6, 1) = '/'
  AND CAST(substr(dt, 1, 2) AS INTEGER) BETWEEN 1 AND 31
  AND CAST(substr(dt, 4, 2) AS INTEGER) BETWEEN 1 AND 12
  AND CAST(substr(dt, 7, 4) AS INTEGER) BETWEEN 2013 AND 2023;
""")
print("tempo povoado.")

# ------------------------------------------------------------
# 4. Povoar municipio
# ------------------------------------------------------------
con.executescript("""
INSERT OR IGNORE INTO municipio (cod_mun, cod_uf, nome_uf)
SELECT DISTINCT
    printf('%06d', CAST(mun_diag AS INTEGER)),
    printf('%02d', CAST(uf_diagn AS INTEGER)),
    NULL
FROM bruto_cancer
WHERE mun_diag IS NOT NULL AND TRIM(mun_diag) <> ''
  AND uf_diagn IS NOT NULL AND TRIM(uf_diagn) <> '';
""")
print("municipio povoado.")

# ------------------------------------------------------------
# 5. Povoar caso_cancer (Opção 2: GROUP BY = UNIQUE)
# ------------------------------------------------------------
con.executescript("""
INSERT INTO caso_cancer (
    id_tempo_diag, id_tempo_trat, mun_diag, cnes_diag,
    diag_deth, idade, sexo, estadiamento,
    qtd_procedimentos, tempo_trat, dt_nasc
)
SELECT
    td.id_tempo                                   AS id_tempo_diag,
    MIN(tt.id_tempo)                              AS id_tempo_trat,
    printf('%06d', CAST(b.mun_diag AS INTEGER))   AS mun_diag,
    printf('%d',   CAST(b.cnes_diag AS INTEGER))  AS cnes_diag,
    b.diag_deth                                   AS diag_deth,
    CAST(b.idade AS INTEGER)                      AS idade,
    b.sexo                                        AS sexo,
    MIN(
        CASE
            WHEN b.estadiam IS NULL OR TRIM(b.estadiam) = '' THEN NULL
            WHEN CAST(b.estadiam AS INTEGER) = 9 THEN NULL
            ELSE CAST(b.estadiam AS INTEGER)
        END
    )                                             AS estadiamento,
    COUNT(*)                                      AS qtd_procedimentos,
    MIN(
        CASE
            WHEN b.tempo_trat IS NULL OR TRIM(b.tempo_trat) = '' THEN NULL
            WHEN CAST(b.tempo_trat AS INTEGER) = 99999 THEN NULL
            WHEN CAST(b.tempo_trat AS INTEGER) < 0 THEN NULL
            ELSE CAST(b.tempo_trat AS INTEGER)
        END
    )                                             AS tempo_trat,
    CASE
        WHEN b.dt_nasc IS NULL OR TRIM(b.dt_nasc) = '' THEN NULL
        ELSE substr(b.dt_nasc, 7, 4) || '-' || substr(b.dt_nasc, 4, 2) || '-' || substr(b.dt_nasc, 1, 2)
    END                                           AS dt_nasc
FROM (
    SELECT *,
        CASE
            WHEN dt_trat IS NULL OR TRIM(dt_trat) = '' THEN NULL
            ELSE substr(dt_trat, 7, 4) || '-' || substr(dt_trat, 4, 2) || '-' || substr(dt_trat, 1, 2)
        END AS dt_trat_iso
    FROM bruto_cancer
) b
LEFT JOIN tempo td
       ON td.data = substr(b.dt_diag, 7, 4) || '-' || substr(b.dt_diag, 4, 2) || '-' || substr(b.dt_diag, 1, 2)
LEFT JOIN tempo tt
       ON tt.data = b.dt_trat_iso
WHERE b.dt_diag IS NOT NULL AND TRIM(b.dt_diag) <> ''
  AND LENGTH(b.dt_diag) = 10
  AND CAST(substr(b.dt_diag, 1, 2) AS INTEGER) BETWEEN 1 AND 31
  AND CAST(substr(b.dt_diag, 4, 2) AS INTEGER) BETWEEN 1 AND 12
  AND CAST(substr(b.dt_diag, 7, 4) AS INTEGER) BETWEEN 2013 AND 2023
  AND b.mun_diag IS NOT NULL AND TRIM(b.mun_diag) <> ''
  AND b.cnes_diag IS NOT NULL AND TRIM(b.cnes_diag) <> ''
GROUP BY
    td.id_tempo,
    b.mun_diag,
    b.cnes_diag,
    b.diag_deth,
    b.idade,
    b.sexo;
""")
con.commit()
print("caso_cancer povoado.")

# ------------------------------------------------------------
# 6. Conferir
# ------------------------------------------------------------
print("\n=== Contagens no modelo ===")
print(pd.read_sql_query("""
    SELECT
        (SELECT COUNT(*) FROM tempo)       AS tempo,
        (SELECT COUNT(*) FROM municipio)   AS municipio,
        (SELECT COUNT(*) FROM caso_cancer) AS caso_cancer
""", con))

con.commit()
con.close()

Tabelas criadas.
tempo povoado.
municipio povoado.
caso_cancer povoado.

=== Contagens no modelo ===
   tempo  municipio  caso_cancer
0   3224        585       613067


In [ ]:
import pandas as pd
import sqlite3

con = sqlite3.connect('dados/base.db')

verificacoes = {
    "1_total_staging_vs_modelo": """
        SELECT
            (SELECT COUNT(*) FROM bruto_cancer) AS staging,
            (SELECT COUNT(*) FROM caso_cancer)  AS casos
    """,

    "2_cns_nulos": """
        SELECT COUNT(*) AS n
        FROM bruto_cancer
        WHERE cns_pac IS NULL OR TRIM(cns_pac) = ''
    """,

    "3_orfaos_municipio": """
        SELECT COUNT(*) AS n
        FROM caso_cancer c
        LEFT JOIN municipio m ON c.mun_diag = m.cod_mun
        WHERE m.cod_mun IS NULL
    """,

    "4_nulos_por_coluna": """
        SELECT
            SUM(CASE WHEN id_tempo_trat IS NULL THEN 1 ELSE 0 END) AS sem_data_trat,
            SUM(CASE WHEN estadiamento  IS NULL THEN 1 ELSE 0 END) AS sem_estadiamento,
            SUM(CASE WHEN tempo_trat     IS NULL THEN 1 ELSE 0 END) AS sem_tempo_trat
        FROM caso_cancer
    """,

    "5_grao_unico": """
        SELECT id_tempo_diag, mun_diag, cnes_diag, diag_deth, idade, sexo, COUNT(*) AS qtd
        FROM caso_cancer
        GROUP BY id_tempo_diag, mun_diag, cnes_diag, diag_deth, idade, sexo
        HAVING COUNT(*) > 1
        LIMIT 20
    """,

    "6_valores_impossiveis": """
        SELECT
            MIN(idade) AS idade_min, MAX(idade) AS idade_max,
            MIN(tempo_trat) AS tempo_min, MAX(tempo_trat) AS tempo_max,
            MIN(qtd_procedimentos) AS qtd_min, MAX(qtd_procedimentos) AS qtd_max
        FROM caso_cancer
    """,
}

for nome, sql in verificacoes.items():
    print(f"\n--- {nome} ---")
    print(pd.read_sql_query(sql, con))

con.commit()
con.close()


--- 1_total_staging_vs_modelo ---
   staging   casos
0   635713  613067

--- 2_cns_nulos ---
        n
0  635713

--- 3_orfaos_municipio ---
   n
0  0

--- 4_nulos_por_coluna ---
   sem_data_trat  sem_estadiamento  sem_tempo_trat
0         310747            332639          304177

--- 5_grao_unico ---
Empty DataFrame
Columns: [id_tempo_diag, mun_diag, cnes_diag, diag_deth, idade, sexo, qtd]
Index: []

--- 6_valores_impossiveis ---
   idade_min  idade_max  tempo_min  tempo_max  qtd_min  qtd_max
0          0        122          0       1631        1       20


In [ ]:
%%writefile .gitignore
*.db
dados/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore
